# Stage 2 Inference: composed [V]+[D] generation

Generates images by composing the Stage 1 adapter ([V], product identity)
and the Stage 2 adapter ([D], defect appearance) on vanilla SD 1.5



## Cell 1 - GPU check

In [ ]:
import torch, sys, os

print(f'Python  : {sys.version[:6]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    raise RuntimeError('No GPU found. Enable GPU in notebook settings before continuing.')

## Cell 2 - Install dependencies
Same pinned versions confirmed working during Stage 2 training.

In [ ]:
import subprocess, sys


# Clean slate: upgrade pip and install a precise, compatible stack 
packages = [
    "pip",
    "numpy==1.26.4",             # numpy <2 to avoid potential breaks
    "diffusers==0.30.0",
    "transformers==4.44.0",
    "accelerate==0.33.0",
    "peft==0.12.0",              # should handle both stage 1 (newer peft) and stage 2 adapters
    "safetensors",
    "Pillow",
    "tqdm",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages
])

# Optional: verify versions
import diffusers, transformers, accelerate, peft
print(f"diffusers {diffusers.__version__}, transformers {transformers.__version__}, "
      f"accelerate {accelerate.__version__}, peft {peft.__version__}")

## Cell 3 - NumPy patch

In [ ]:
import numpy as np

if not hasattr(np, 'float_'):
    np.float_ = np.float64
if not hasattr(np, 'complex_'):
    np.complex_ = np.complex128

print(f'NumPy {np.__version__} - patch applied.')

## Cell 4 - HuggingFace auth
Needed to download SD 1.5

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets  = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
login(token=hf_token)
print('HuggingFace : OK')

## Cell 5 - Clone repo and set working directory
To obtain the latest `inference/inference.py` version

In [ ]:
import os
from pathlib import Path

WORK     = '/kaggle/working'
REPO_URL = 'https://github.com/alecanc/Few-Shot-Personalization-of-a-Diffusion-Model-for-Industrial-Defect-Synthesis'
REPO_DIR = f'{WORK}/repo'

os.makedirs(f'{WORK}/generated', exist_ok=True)

if not Path(REPO_DIR).exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
CONFIG_PATH = f'{REPO_DIR}/defect-synthesis/config.yaml'
print(f'Working directory : {os.getcwd()}')

## Cell 6 - Patch config.yaml with runtime paths
Patches model_id and output path

In [ ]:
import yaml

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

cfg['model_id']             = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
cfg['paths']['generated']   = f'{WORK}/generated'
cfg['paths']['output_root'] = WORK

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f"model_id  : {cfg['model_id']}")
print(f"generated : {cfg['paths']['generated']}")
print(f"token_V   : {cfg['token_V']}  |  token_D : {cfg['token_D']}")
print(f"weight_V  : {cfg['inference']['adapter_weight_V']}  |  weight_D : {cfg['inference']['adapter_weight_D']}")

## Cell 7 - Verify adapter paths

Set the adapter directories

In [ ]:
from pathlib import Path

# ==== To be edited each run ================================================
CATEGORY    = 'bottle'
DEFECT_TYPE = 'broken_large'

STAGE1_DIR  = f'/kaggle/input/datasets/eliavige/mvtec-adapters-v1/stage1/{CATEGORY}/final'
STAGE2_DIR  = f'/kaggle/input/datasets/eliavige/mvtec-adapters-v1/stage2/{CATEGORY}/{DEFECT_TYPE}/final'

# Adapter weights to override config.yaml if desired
# Config defaults: weight_V=1.0, weight_D=0.8
WEIGHT_V = 1.0
WEIGHT_D = 0.8

# set to None to use the default prompt
PROMPT = None

# ===========================================================================

all_ok = True
for label, path in [('Stage 1', STAGE1_DIR), ('Stage 2', STAGE2_DIR)]:
    p  = Path(path)
    st = next(p.glob('*.safetensors'), None) if p.exists() else None
    if st is None:
        print(f'  x!  {label} - NOT FOUND at {path}')
        all_ok = False
    else:
        print(f'  ok  {label} - {st.name}  ({st.stat().st_size/1e6:.1f} MB)')

if not all_ok:
    raise FileNotFoundError(
        'One or more adapters missing.\n'
        'Check STAGE1_DIR and STAGE2_DIR above against the Data sidebar paths.'
    )
print(f'\nReady: {CATEGORY} / {DEFECT_TYPE} | V={WEIGHT_V} D={WEIGHT_D}')

# === Override inference parameters  =====================================
#cfg['inference']['guidance_scale']       = 7.5   # default 7.5
#cfg['inference']['num_inference_steps']  = 30     # default 30
#cfg['seed']                              = 42    # default 42
# ========================================================================

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f"guidance_scale : {cfg['inference']['guidance_scale']}")
print(f"steps          : {cfg['inference']['num_inference_steps']}")
print(f"seed           : {cfg['seed']}")

## Iterate over combination of weights

In [ ]:
v_weights = [0.2, 0.5, 0.8, 1.0]
d_weights = [0.5, 0.75, 1.0]

total = len(v_weights) * len(d_weights)
done  = 0

for wv in v_weights:
    for wd in d_weights:
        done += 1
        print(f"\n[{done}/{total}] V={wv}  D={wd}")
        !python -u inference/inference_new.py \
            --config      {CONFIG_PATH} \
            --category    {CATEGORY} \
            --defect_type {DEFECT_TYPE} \
            --stage1_dir  {STAGE1_DIR} \
            --stage2_dir  {STAGE2_DIR} \
            --weight_v    {wv} \
            --weight_d    {wd} \
            {"--prompt" if PROMPT else ""} {f'"{PROMPT}"' if PROMPT else ""}

Save all images in a zip

In [ ]:
import shutil
from pathlib import Path

zip_base = f"{WORK}/weight_sweep_{CATEGORY}_{DEFECT_TYPE}"
shutil.make_archive(zip_base, "zip", f"{WORK}/generated")
zip_path = Path(f"{zip_base}.zip")
print(f"zip: {zip_path.name}  ({zip_path.stat().st_size/1e6:.1f} MB)")
print("download from the Output panel in the right sidebar")

## Cell 8 - Run inference

Calls `inference.py` with the paths and weights set above.
Generates `num_images_per_prompt` images (from config, default 8) and saves a 4×2 grid


In [ ]:
!python -u inference/inference_new.py \
    --config      {CONFIG_PATH} \
    --category    {CATEGORY} \
    --defect_type {DEFECT_TYPE} \
    --stage1_dir  {STAGE1_DIR} \
    --stage2_dir  {STAGE2_DIR} \
    --weight_v    {WEIGHT_V} \
    --weight_d    {WEIGHT_D}

## Cell 9 - Display result
Shows the generated grid inline and confirms the output path.

In [ ]:
from pathlib import Path
from PIL import Image as PILImage
from IPython.display import display
import yaml

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

n_images  = cfg['inference']['num_images_per_prompt']
grid_path = Path(f"{WORK}/generated/{CATEGORY}_{DEFECT_TYPE}_composed_{n_images}imgs.png")

if not grid_path.exists():
    raise FileNotFoundError(
        f'Grid not found: {grid_path}\n'
        f'Check the output of Cell 8 for errors.'
    )

print(f'Grid: {grid_path.name}')
display(PILImage.open(grid_path))

## Cell 10 - Run different weights combination:

Re-run inference with different adapter weights to see their effect without
reloading SD 1.5, just change the weights and params in Cell 7 and re-run from Cell 8


! Attnetion to not overwrite the images

## Cell 11 - Save output
Zips the generated folder, ready to download from the Output panel

In [ ]:
import shutil
from pathlib import Path

zip_base = f'{WORK}/inference_{CATEGORY}_{DEFECT_TYPE}'
shutil.make_archive(zip_base, 'zip', f'{WORK}/generated')
zip_path = Path(f'{zip_base}.zip')
print(f'Zip: {zip_path.name}  ({zip_path.stat().st_size/1e6:.1f} MB)')
print('Download from the Output panel in the right sidebar')